<a href="https://colab.research.google.com/github/Ehsan-Roohi/Introduction-to-Compressible-Flows/blob/main/notebooks/chapter07/07_02_interacting_shock_pressure_ratios.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Pressure Ratio After Interacting Shock Waves

**Book:** *Introduction to Compressible Flows*<br>
**Chapter 7:** Unsteady Wave Motion<br>
**Concepts:** shock interaction · implicit pressure matching · Brent root solve

## Learning objectives

- Form the nonlinear compatibility equation for sequential shocks.
- Solve for the final pressure ratio with a robust bracketed method.
- Trace local pressure ratios into a global p5/p1 value.

## How to use this notebook

Set p2/p1 and p3/p2 in the parameter block and run the cell.

> **Model scope.** The implemented relation is specialized to the pressure-ratio configuration derived in Chapter 7 and assumes a perfect gas.


In [ ]:
import numpy as np
from scipy.optimize import root_scalar
gamma   = 1.4
alpha1  = (gamma + 1.0) / (gamma - 1.0)
beta1   = (gamma - 1.0) / (2*gamma)
p2_p1   = 3.0      # p2/p1
p3_p2   = 3.0      # p3/p2
p1_p2 = 1.0 / p2_p1
p1_p3 = 1.0 / (p2_p1 * p3_p2)  # = p1/p3
def f(z):
    pref = np.sqrt(
        beta1 * (1 + alpha1 * p3_p2) /
        ((alpha1 + p3_p2) * (1 + alpha1 * p1_p2) * p3_p2)
    )
    term1 = (z - p1_p3) * np.sqrt((alpha1 + p1_p2) * p3_p2 / (p1_p3 + alpha1 * z))
    term2 = (p3_p2 - 1.0) * np.sqrt((alpha1 * p1_p2 + 1.0) / (1.0 + alpha1 * p3_p2))
    term3 = (1.0 - p1_p2)
    return pref * (term1 - term2 - term3) + z**beta1 - 1.0
sol = root_scalar(
    f,
    bracket=[1e-8, 10],
    method='brentq',
    xtol=1e-8,
    rtol=1e-12,
    maxiter=100
)
if sol.converged:
    z = sol.root
    print(f"✔️ Converged: p4/p3 = {z:.8f}")
    p5_p1 = p2_p1 * p3_p2 * z
    print(f"🧮 Final pressure ratio: p5/p1 = {p5_p1:.8f}")
else:
    print("❌ Not converged. You may need to adjust the range or parameters.")


## Interpretation and verification checklist

- Confirm that the chosen root bracket contains a sign change.
- Recompute p1/p2 and p1/p3 whenever either prescribed ratio changes.
- Check that all returned pressure ratios are positive.

## Reproducibility note

Restart the runtime and run the notebook from top to bottom after changing inputs. Record the input values, units, heat-capacity ratio, numerical tolerances, and dependency versions with any result used in coursework, research, or a publication.

Questions or reproducibility reports are welcome through the repository's issue templates.
